# Step 2: Clean Competition Data

Input: `data/0_raw/competition/wide_all.csv` (stacked output of Step 1).

Outputs:

- `data/1_derived/competition/wide_clean.csv`: one row per participant, with utility columns, design factors, and four binary hypothesis indicators.
- `data/1_derived/competition/long_utils.csv`: one row per (participant, rank), with utility.

## Cleaning steps

The pipeline runs four steps in order, producing the final analysis samples of 1960, 1001, and 423 participants for Studies 1, 2, and 3.

### Step 1: Collection-window filter

Keep only rows whose `dateadded` falls inside each study's data-collection window. Rows before the window are test entries; rows after it are post-collection stimulus checks, often carrying placeholder demographics such as `age = 1`. The window is defined by collection date alone, so demographic values such as age are never used as a filter.

### Step 2: IP deduplication

Applied to Studies 1 and 2 only (Study 3 is a lab sample and is exempt). See the `smart_dedup` cell for the per-IP rules.

### Step 3: Complete-case

Drop any row with a blank in a shown response cell.

### Step 4: Order

Deduplication (Step 2) is applied before the complete-case drop (Step 3).

## Utilities and indicators

Each `rank{N}indif` column is the indifference probability that makes the participant indifferent between "rank N for sure" and a gamble at probability p for 1st vs (1-p) for last. With $U(1st) = 1$ and $U(last) = 0$, this gives $U(N) = p$.

### Six-person studies (exp72, exp73)

Elicited ranks 2, 3, 4, 5; anchors $U(1)=1$, $U(6)=0$.

- H1 (monotone, strict): $U(2) \ge U(3) \ge U(4) \ge U(5)$ and $U(2) > U(5)$.
- H2a (convex at rank 2): $U(2) - U(3) < U(1) - U(2)$.
- H2b (concave at rank 5): $U(5) - U(6) > U(4) - U(5)$.
- H4 (drop from 1st > drop into last): $U(1) - U(2) > U(5) - U(6)$.

### Ten-person study (exp58)

Elicited ranks 2, 3, 5, 8, 9; anchors $U(1)=1$, $U(10)=0$.

- H1: $U(2) \ge U(3) \ge U(5) \ge U(8) \ge U(9)$ and $U(2) > U(9)$.
- H2a: $U(2) - U(3) < U(1) - U(2)$.
- H2b: $U(9) - U(10) > U(8) - U(9)$.
- H4: $U(1) - U(2) > U(9) - U(10)$.

### Design factor encoding

- `wording` from `rlabel`: `w` to `word`, `n` to `numerical`.
- `order` from `ordercon`: 1 to `ascending`, 2 to `descending`.
- `frame` from `domain` (exp72/exp73 only): `i` to `intelligence`, `p` to `physical`.

In [1]:
# Setup: locate the project root (the folder containing run_all.py), then load
# the stacked raw export produced by Step 1. `wide_all.csv` has one row per raw
# submission across all three studies (exp72 = Study 1, exp73 = Study 2,
# exp58 = Study 3), before any cleaning.
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "run_all.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not find project root (run_all.py)")
    ROOT = ROOT.parent

RAW    = ROOT / "data" / "0_raw"     / "competition"
DERIV  = ROOT / "data" / "1_derived" / "competition"
OUTLOG = ROOT / "output" / "1_derived" / "clean_competition"
DERIV.mkdir(parents=True, exist_ok=True)
OUTLOG.mkdir(parents=True, exist_ok=True)

wide = pd.read_csv(RAW / "wide_all.csv")
print(f"loaded wide_all: {wide.shape}")
print(wide['study'].value_counts())

loaded wide_all: (3599, 38)
study
exp72    2088
exp73    1065
exp58     446
Name: count, dtype: int64


## Encode design factors

In [2]:
# Encode the experimental design factors from raw codes into readable labels.
# `frame` only exists for the six-person MTurk studies; exp58 (lab) has no
# `domain` in the raw export and is assigned its domain later from the collection date.
wide["wording"] = wide["rlabel"].map({"w": "word", "n": "numerical"})
wide["order"]   = wide["ordercon"].map({1: "ascending", 2: "descending"})
wide["frame"]   = wide["domain"].map({"i": "intelligence", "p": "physical"}) if "domain" in wide.columns else np.nan

print("wording x study:")
print(pd.crosstab(wide["study"], wide["wording"], dropna=False))
print("\norder x study:")
print(pd.crosstab(wide["study"], wide["order"], dropna=False))
print("\nframe x study:")
print(pd.crosstab(wide["study"], wide["frame"], dropna=False))

wording x study:
wording  numerical  word
study                   
exp58          217   229
exp72         1023  1065
exp73          548   517

order x study:
order  ascending  descending
study                       
exp58        245         201
exp72       1006        1082
exp73        541         524

frame x study:
frame  intelligence  physical  NaN
study                             
exp58             0         0  446
exp72          1058      1030    0
exp73           519       546    0


## Step 1: Collection-window filter (applied first)

Keep only the rows whose `dateadded` timestamp falls inside each study's data-collection window, and drop the rest.

### Windows used

| Study | Window (`dateadded` calendar day) | Rows in window |
|:------|:----------------------------------|---------------:|
| 1 (exp72) | 2017-08-09 (single day) | 2067 |
| 2 (exp73) | 2018-02-01 (single day) | 1054 |
| 3 (exp58, lab) | 2015-01-19 to 2015-02-06 (physical wave) plus 2015-03-09 (intellectual wave) | 428 |

For Study 3 the window is the two collection waves only. Four isolated straggler days (`2015-02-11`, `2015-03-03`, `2015-03-06`, `2015-03-18`, each with 1 to 3 rows) fall outside the waves and are excluded; one of them carries an out-of-wave `age = 1` placeholder row (participant ID 23867, dated `2015-03-06`).

In [3]:
# === STEP 1: collection-window filter, applied first ===
# Keep only rows whose `dateadded` calendar day falls inside the study's
# data-collection window.
#
# Normalize dateadded to the calendar day (drop the time-of-day component) so the
# comparison is on the date alone.
_day = pd.to_datetime(wide["dateadded"], errors="coerce").dt.normalize()

# Study 3 physical wave is a date range; intellectual wave is a single day.
_s3_physical     = (_day >= pd.Timestamp("2015-01-19")) & (_day <= pd.Timestamp("2015-02-06"))
_s3_intellectual = (_day == pd.Timestamp("2015-03-09"))

in_window = (
    ((wide["study"] == "exp72") & (_day == pd.Timestamp("2017-08-09")))           # Study 1: single day
    | ((wide["study"] == "exp73") & (_day == pd.Timestamp("2018-02-01")))         # Study 2: single day
    | ((wide["study"] == "exp58") & (_s3_physical | _s3_intellectual))            # Study 3: two waves
)

before = wide["study"].value_counts()
wide = wide[in_window].copy()
after = wide["study"].value_counts()
for s in ["exp72", "exp73", "exp58"]:
    drop = before.get(s, 0) - after.get(s, 0)
    print(f"  {s}: {before.get(s, 0):>4} -> {after.get(s, 0):>4}  "
          f"(dropped {drop} out-of-window; kept {after.get(s, 0)} in-window)")
# In-window counts: 2067 / 1054 / 428 for Studies 1 / 2 / 3.
# =============================================================================

  exp72: 2088 -> 2067  (dropped 21 out-of-window; kept 2067 in-window)
  exp73: 1065 -> 1054  (dropped 11 out-of-window; kept 1054 in-window)
  exp58:  446 ->  428  (dropped 18 out-of-window; kept 428 in-window)


## Step 2: IP deduplication (`smart_dedup`, Studies 1 and 2 only)

Identify respondents who share an IP address and keep only the first, using the per-IP rules below. For each IP, within a study:

1. Single row: keep it.
2. Multiple rows with different demographics (a real shared household or connection): keep the first chronologically, capped per study (Study 1: 1; Study 2: 2).
3. Multiple rows that are entirely one person (identical `gender`, `age`, `ethnicity`): if all attempts share the same `rlabel` (same experimental cell, a repeated attempt), drop the whole IP; if `rlabel` differs, keep the first chronologically (capped).

Study 3 (exp58) is a lab sample and is exempt from IP deduplication.

Deduplication runs only on rows already inside the collection window (Step 1), so a repeat IP only matters when both rows are in-window.

In [4]:
# === STEP 2: IP deduplication, rlabel-aware (smart_dedup) ===
# Complete-case is applied in a later cell so that Study 1 is deduplicated
# before complete-case (the order Study 1 requires).
#
# Rule, per IP within a study:
#   1. Single row: keep.
#   2. Multi-row, DIFFERENT (gender, age, ethnicity) = real household:
#        keep the chronologically first row, capped per study.
#   3. Multi-row, IDENTICAL (gender, age, ethnicity) = same person re-attempting:
#        a) SAME rlabel across attempts (same experimental cell):
#             drop the entire IP.
#        b) DIFFERENT rlabel (independent observations):
#             keep the chronologically first row, capped per study.
#   Cap per study: Study 1 (exp72) = 1; Study 2 (exp73) = 2.
#   Study 3 (exp58): no deduplication (lab environment).
#
# For example, IP 69.117.181.144 (participant IDs 31715 / 32605) is one person
# who repeated the same experimental cell, so the whole IP is dropped. IP
# 67.183.210.112 holds three different people (males aged 26, 27, and 28), so it
# takes the household branch (rule 2) and keeps the first respondent
# (participant ID 32247).
import pandas as pd

def smart_dedup(df, study_code, max_per_household):
    """Return a list of indices to KEEP for this study."""
    s = df[df['study'] == study_code]
    keep_idxs = []
    for ip, grp in s.groupby('ip'):
        if len(grp) == 1:
            keep_idxs.extend(grp.index.tolist())
            continue
        demo = grp[['gender','age','ethnicity']].fillna('__NA__')
        if demo.drop_duplicates().shape[0] == 1:
            # Same person cluster
            if grp['rlabel'].nunique() == 1:
                # Same experimental cell, dependent re-evaluation, drop all
                continue
            # Different cell, independent observations, keep the first chronologically
            sub = grp.sort_values('dateadded').head(max_per_household)
            keep_idxs.extend(sub.index.tolist())
        else:
            # Different demographics = real household
            sub = grp.sort_values('dateadded').head(max_per_household)
            keep_idxs.extend(sub.index.tolist())
    return keep_idxs

before = wide['study'].value_counts()

# Study 1 (exp72): smart deduplication, cap 1 per IP. No age filter.
mask_s1 = (wide['study'] == 'exp72') & (wide['n_ranks'] == 6)
s1 = wide.loc[smart_dedup(wide.loc[mask_s1], 'exp72', max_per_household=1)].copy()

# Study 2 (exp73): smart deduplication, cap 2 per IP.
mask_s2 = (wide['study'] == 'exp73') & (wide['n_ranks'] == 6)
s2 = wide.loc[smart_dedup(wide.loc[mask_s2], 'exp73', max_per_household=2)].copy()

# Study 3 (exp58): no deduplication (lab environment).
mask_s3 = wide['n_ranks'] == 10
s3 = wide.loc[mask_s3]

wide = pd.concat([s1, s2, s3], ignore_index=False).sort_index()
after = wide['study'].value_counts()
print(f"  exp72: {before.get('exp72', 0):>4} -> {after.get('exp72', 0):>4}  (smart deduplication, cap 1)")
print(f"  exp73: {before.get('exp73', 0):>4} -> {after.get('exp73', 0):>4}  (smart deduplication, cap 2)")
print(f"  exp58: {before.get('exp58', 0):>4} -> {after.get('exp58', 0):>4}  (no deduplication; lab environment)")
# Complete-case is applied next (separate cell), which keeps Study 1's required
# deduplication-before-complete-case ordering. See the next markdown cell.
# =============================================================================

  exp72: 2067 -> 2015  (smart deduplication, cap 1)
  exp73: 1054 -> 1050  (smart deduplication, cap 2)
  exp58:  428 ->  428  (no deduplication; lab environment)


## Step 3: Complete-case, then build utility columns

Drop participants with incomplete responses. A row is dropped if it has a blank in any shown response cell, not only the elicited indifference-probability columns.

### Which cells count as shown response cells

Each respondent answered the elicited indifference-probability questions plus a fixed block of attitude and demographic items. Every retained participant has a complete block, so the complete-case drop covers all of it:

- Six-person studies (exp72, exp73): `rank2indif`, `rank3indif`, `rank4indif`, `rank5indif`, `feeldrop`, `feelrise`, `howcompetitive`, `howfitm`, `howfitp`, `gender`, `age`, `ethnicity`, `actuallyrank`.
- Ten-person study (exp58, lab): `rank2indif`, `rank3indif`, `rank5indif`, `rank8indif`, `rank9indif`, `howcompetitive`, `howfit`, `gender`, `age`, `actuallyrank`. (No `ethnicity` column in the lab sample, and the lab version used `howfit` rather than the `howfitm`/`howfitp` pair.)

### Order: deduplication before complete-case

Deduplication (Step 2) runs before this complete-case step. 

In [5]:
# === Build utility columns from the elicited indifference-probability cells ==
ELICITED = {
    6:  [2, 3, 4, 5],         # exp72, exp73
    10: [2, 3, 5, 8, 9],      # exp58 (rank 5 is also elicited)
}

def utility_cols_for_row(row):
    # U(1)=1 and U(last)=0 are the anchors; elicited ranks read their indifference
    # probability directly.
    n = int(row["n_ranks"])
    u = {f"u{r}": np.nan for r in range(1, n + 1)}
    u["u1"] = 1.0
    u[f"u{n}"] = 0.0
    for r in ELICITED[n]:
        u[f"u{r}"] = row.get(f"rank{r}indif", np.nan)
    return pd.Series(u)

wide = pd.concat([wide, wide.apply(utility_cols_for_row, axis=1)], axis=1)

# === STEP 3: complete-case on the shown response cells =======================
# A row is complete only if it has no blank in ANY shown response cell, which is
# the elicited indifference-probability columns PLUS the fixed attitude and
# demographic block that every respondent answered. The cell set differs by
# study (see the markdown above): the lab study (exp58) has no `ethnicity` and
# uses `howfit` instead of the `howfitm`/`howfitp` pair.
COMPLETE_CASE_COLS = {
    "exp72": [f"rank{r}indif" for r in ELICITED[6]]
             + ["feeldrop", "feelrise", "howcompetitive", "howfitm", "howfitp",
                "gender", "age", "ethnicity", "actuallyrank"],
    "exp73": [f"rank{r}indif" for r in ELICITED[6]]
             + ["feeldrop", "feelrise", "howcompetitive", "howfitm", "howfitp",
                "gender", "age", "ethnicity", "actuallyrank"],
    "exp58": [f"rank{r}indif" for r in ELICITED[10]]
             + ["howcompetitive", "howfit", "gender", "age", "actuallyrank"],
}

def is_complete(row):
    cols = [c for c in COMPLETE_CASE_COLS[row["study"]] if c in wide.columns]
    return row[cols].notna().all()

wide["complete"] = wide.apply(is_complete, axis=1)
n_drop = (~wide["complete"]).sum()
print(f"Dropping {n_drop} incomplete participants (blank in a shown response cell)")
print(wide.groupby("study")["complete"].agg(["sum", "count"]))

wide = wide.loc[wide["complete"]].drop(columns=["complete"]).copy()
print(f"\nFinal N: {wide.shape[0]}")
print(wide['study'].value_counts())
print("\nExpected final N: exp72=1960, exp73=1001, exp58=423")

Dropping 109 incomplete participants (blank in a shown response cell)
        sum  count
study             
exp58   423    428
exp72  1960   2015
exp73  1001   1050

Final N: 3384
study
exp72    1960
exp73    1001
exp58     423
Name: count, dtype: int64

Expected final N: exp72=1960, exp73=1001, exp58=423


## Build hypothesis indicators

In [6]:
# Build the four binary hypothesis indicators (H1, H2a, H2b, H4).
def six_rank_indicators(df):
    u1, u2, u3, u4, u5, u6 = df["u1"], df["u2"], df["u3"], df["u4"], df["u5"], df["u6"]
    # H4 compares utility drops at 2-decimal precision, so round before comparing.
    # Without rounding, `1 - 0.98 > 0.02` is True in floating point because
    # `1 - 0.98 = 0.020000000000000018`.
    return pd.DataFrame({
        "H1":  (u2 >= u3) & (u3 >= u4) & (u4 >= u5) & (u2 > u5),
        "H2a": (u2 - u3) < (u1 - u2),
        "H2b": (u5 - u6) > (u4 - u5),
        "H4":  (u1 - u2).round(2) > (u5 - u6).round(2),
    }, index=df.index)

def ten_rank_indicators(df):
    u1, u2, u3, u5, u8, u9, u10 = df["u1"], df["u2"], df["u3"], df["u5"], df["u8"], df["u9"], df["u10"]
    return pd.DataFrame({
        "H1":  (u2 >= u3) & (u3 >= u5) & (u5 >= u8) & (u8 >= u9) & (u2 > u9),
        "H2a": (u2 - u3) < (u1 - u2),
        "H2b": (u9 - u10) > (u8 - u9),
        "H4":  (u1 - u2).round(2) > (u9 - u10).round(2),
    }, index=df.index)

ind_six = six_rank_indicators(wide[wide["n_ranks"] == 6])
ind_ten = ten_rank_indicators(wide[wide["n_ranks"] == 10])
wide = wide.join(pd.concat([ind_six, ind_ten]).sort_index())

# Study 3 `domain` is set from the collection wave. exp58 was fielded as two
# experiments, a physical-domain run (Jan 19 to Feb 6, 2015) and an
# intellectual-domain run (March 9, 2015); the `domain` column is not in the raw
# export, so it is assigned from `dateadded`: the physical wave is `p` (302
# participants), the intellectual wave is `i` (121 participants).
s3_mask = wide["study"] == "exp58"
s3_dates = pd.to_datetime(wide.loc[s3_mask, "dateadded"], errors="coerce")
wide.loc[s3_mask, "domain"] = (s3_dates >= "2015-03-01").map({True: "i", False: "p"}).values

print("Indicator prevalence by study (%):")
for h in ["H1", "H2a", "H2b", "H4"]:
    pct = wide.groupby("study")[h].mean() * 100
    print(f"  {h}: " + "  ".join(f"{s}={pct[s]:.1f}" for s in pct.index))

Indicator prevalence by study (%):
  H1: exp58=58.2  exp72=71.9  exp73=69.1
  H2a: exp58=79.2  exp72=79.3  exp73=80.3
  H2b: exp58=67.6  exp72=72.2  exp73=68.0
  H4: exp58=70.4  exp72=68.0  exp73=68.8


## Build long utilities (one row per participant × rank, anchors + elicited only)

In [7]:
# Reshape to long form: one row per (participant, rank), keeping only the anchor
# ranks (1 and last) plus the elicited ranks. Convenient input for Figure 1.
long_rows = []
for _, row in wide.iterrows():
    n = int(row["n_ranks"])
    for r in [1] + ELICITED[n] + [n]:
        long_rows.append({
            "study":   row["study"],
            "pid":     row["pid"],
            "n_ranks": n,
            "rank":    r,
            "utility": row[f"u{r}"],
        })
long_utils = pd.DataFrame(long_rows)
print(f"long_utils: {long_utils.shape}")

long_utils: (20727, 5)


## Write outputs

In [8]:
# Write the two cleaned outputs (wide, one row per participant; long, one row per
# participant-rank) plus a small summary log. Downstream analysis scripts read
# these CSVs.
wide_out = DERIV / "wide_clean.csv"
long_out = DERIV / "long_utils.csv"
wide.to_csv(wide_out, index=False)
long_utils.to_csv(long_out, index=False)
print(f"wrote {wide_out.relative_to(ROOT)}  ({wide.shape[0]} rows, {wide.shape[1]} cols)")
print(f"wrote {long_out.relative_to(ROOT)}  ({long_utils.shape[0]} rows)")

log = OUTLOG / "clean_summary.txt"
with open(log, "w") as f:
    f.write(f"After clean: {wide.shape[0]} participants\n")
    f.write("By study:\n")
    f.write(wide['study'].value_counts().to_string() + "\n\n")
    f.write("Indicator prevalence (%):\n")
    for h in ["H1", "H2a", "H2b", "H4"]:
        f.write(f"  {h}:\n{(wide.groupby('study')[h].mean()*100).round(1).to_string()}\n")
print(f"wrote {log.relative_to(ROOT)}")

wrote data\1_derived\competition\wide_clean.csv  (3384 rows, 55 cols)
wrote data\1_derived\competition\long_utils.csv  (20727 rows)
wrote output\1_derived\clean_competition\clean_summary.txt
